# Model 000: Dumb Baselines

This notebook builds simple baseline models for:

- `recommended` (binary sentiment target)
- `votes_helpful` (count helpfulness target)
- `is_helpful` (binary helpfulness target, derived as `votes_helpful >= 1`)

These baselines are intentionally simple and set a floor that stronger models should beat.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, mean_absolute_error, mean_squared_error

SEED = 2026
rng = np.random.default_rng(SEED)

base_dir = Path("../../data/processed")
train_path = base_dir / "steam_reviews_cleaned_english_train.parquet"
val_path = base_dir / "steam_reviews_cleaned_english_val.parquet"
# test_path = base_dir / "steam_reviews_cleaned_english_test.parquet"

for p in (train_path, val_path):
    if not p.exists():
        raise FileNotFoundError(f"Missing split parquet: {p}")

print("Train:", train_path)
print("Val:  ", val_path)
# print("Test: ", test_path)

Train: ../../data/processed/steam_reviews_cleaned_english_train.parquet
Val:   ../../data/processed/steam_reviews_cleaned_english_val.parquet


In [2]:
train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
# test_df = pd.read_parquet(test_path)

for df in (train_df, val_df):
    if "is_helpful" not in df.columns:
        df["is_helpful"] = df["votes_helpful"] >= 1

print(f"train shape: {train_df.shape}")
print(f"val shape:   {val_df.shape}")
# print(f"test shape:  {test_df.shape}")

train shape: (6410756, 19)
val shape:   (1374737, 19)


In [3]:
def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }


def regression_metrics(y_true, y_pred):
    # Compatible with older sklearn versions that do not support squared=False.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return {"rmse": rmse, "mae": mae}


def as_table(rows):
    return pd.DataFrame(rows).sort_values(["target", "baseline"]).reset_index(drop=True)

In [4]:
# ---- Baselines for recommended (binary) ----
p_rec = train_df["recommended"].mean()
majority_rec = train_df["recommended"].mode().iloc[0]

recommended_rows = []
for split_name, split_df in [("train", train_df), ("val", val_df)]:
    y_true = split_df["recommended"].astype(int)

    # Majority class baseline
    y_pred_majority = np.full(len(split_df), int(majority_rec), dtype=int)
    m = classification_metrics(y_true, y_pred_majority)
    recommended_rows.append({"target": "recommended", "split": split_name, "baseline": "majority", **m})

    # Prevalence-random baseline
    y_pred_rand = rng.binomial(1, p_rec, size=len(split_df))
    m = classification_metrics(y_true, y_pred_rand)
    recommended_rows.append({"target": "recommended", "split": split_name, "baseline": "prevalence_random", **m})

pd.DataFrame(recommended_rows)

,target,split,baseline,accuracy,f1,precision,recall
0,recommended,train,majority,0.887713,0.940517,0.887713,1.000000
1,recommended,train,prevalence_random,0.800847,0.887841,0.887740,0.887942
2,recommended,val,majority,0.887523,0.940410,0.887523,1.000000
3,recommended,val,prevalence_random,0.800352,0.887528,0.887507,0.887549


In [5]:
# ---- Baselines for votes_helpful (count / regression-style) ----
helpful_train = train_df["votes_helpful"].astype(float)
mean_helpful = helpful_train.mean()

# empirical distribution sampler from train
value_probs = helpful_train.value_counts(normalize=True).sort_index()
values = value_probs.index.to_numpy()
probs = value_probs.values

votes_rows = []
for split_name, split_df in [("train", train_df), ("val", val_df)]:
    y_true = split_df["votes_helpful"].astype(float).to_numpy()

    # predict all zeros
    y_pred_zero = np.zeros(len(split_df), dtype=float)
    m = regression_metrics(y_true, y_pred_zero)
    votes_rows.append({"target": "votes_helpful", "split": split_name, "baseline": "all_zero", **m})

    # predict train mean
    y_pred_mean = np.full(len(split_df), mean_helpful, dtype=float)
    m = regression_metrics(y_true, y_pred_mean)
    votes_rows.append({"target": "votes_helpful", "split": split_name, "baseline": "train_mean", **m})

    # sample from train empirical distribution
    y_pred_emp = rng.choice(values, size=len(split_df), p=probs)
    m = regression_metrics(y_true, y_pred_emp)
    votes_rows.append({"target": "votes_helpful", "split": split_name, "baseline": "empirical_random", **m})

pd.DataFrame(votes_rows)

,target,split,baseline,rmse,mae
0,votes_helpful,train,all_zero,46.488701,2.011896
1,votes_helpful,train,train_mean,46.445146,3.173951
2,votes_helpful,train,empirical_random,66.742558,3.801318
3,votes_helpful,val,all_zero,50.281343,2.025414
4,votes_helpful,val,train_mean,50.240535,3.187728
5,votes_helpful,val,empirical_random,64.369496,3.766435


In [8]:
# ---- Baselines for is_helpful (binary helpfulness) ----
p_helpful = train_df["is_helpful"].mean()
majority_helpful = train_df["is_helpful"].mode().iloc[0]

is_helpful_rows = []
for split_name, split_df in [("train", train_df), ("val", val_df)]:
    y_true = split_df["is_helpful"].astype(int)

    y_pred_majority = np.full(len(split_df), int(majority_helpful), dtype=int)
    m = classification_metrics(y_true, y_pred_majority)
    is_helpful_rows.append({"target": "is_helpful", "split": split_name, "baseline": "majority", **m})

    y_pred_rand = rng.binomial(1, p_helpful, size=len(split_df))
    m = classification_metrics(y_true, y_pred_rand)
    is_helpful_rows.append({"target": "is_helpful", "split": split_name, "baseline": "prevalence_random", **m})

pd.DataFrame(is_helpful_rows)

,target,split,baseline,accuracy,f1,precision,recall
0,is_helpful,train,majority,0.701121,0.000000,0.000000,0.000000
1,is_helpful,train,prevalence_random,0.580789,0.298590,0.298635,0.298545
2,is_helpful,val,majority,0.701237,0.000000,0.000000,0.000000
3,is_helpful,val,prevalence_random,0.580221,0.298239,0.297914,0.298565


In [7]:
# Combined quick summary (train + validation) only
summary = pd.concat(
    [
        pd.DataFrame(recommended_rows),
        pd.DataFrame(votes_rows),
        pd.DataFrame(is_helpful_rows),
    ],
    ignore_index=True,
)
summary

,target,split,baseline,accuracy,f1,precision,recall,rmse,mae
0,recommended,train,majority,0.887713,0.940517,0.887713,1.000000,NaN,NaN
1,recommended,train,prevalence_random,0.800847,0.887841,0.887740,0.887942,NaN,NaN
2,recommended,val,majority,0.887523,0.940410,0.887523,1.000000,NaN,NaN
3,recommended,val,prevalence_random,0.800352,0.887528,0.887507,0.887549,NaN,NaN
4,votes_helpful,train,all_zero,NaN,NaN,NaN,NaN,46.488701,2.011896
5,votes_helpful,train,train_mean,NaN,NaN,NaN,NaN,46.445146,3.173951
6,votes_helpful,train,empirical_random,NaN,NaN,NaN,NaN,66.742558,3.801318
7,votes_helpful,val,all_zero,NaN,NaN,NaN,NaN,50.281343,2.025414
8,votes_helpful,val,train_mean,NaN,NaN,NaN,NaN,50.240535,3.187728
9,votes_helpful,val,empirical_random,NaN,NaN,NaN,NaN,64.369496,3.766435
